# retainx walkthrough

Run `dbt build` first and set `DATABASE_URL` (see the README). Run the cells top to bottom; the charts come from your data.
Install the extras with `pip install -r requirements-notebooks.txt`. Start Jupyter from the repo root so `src` imports work.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sqlalchemy import create_engine
from src import config, features, health, personas
from src.eval import ranking_check

engine = create_engine(config.DB_URL)
orders = features.load_orders(engine)
print(f"{len(orders):,} valid orders, {orders['customer_id'].nunique():,} customers, "
      f"{orders['invoice_ts'].min().date()} to {orders['invoice_ts'].max().date()}")

## 1. Revenue is concentrated in few customers

In [ ]:
rev = orders.groupby("customer_id")["order_revenue"].sum().sort_values(ascending=False)
cum = rev.cumsum() / rev.sum()
share = np.arange(1, len(rev) + 1) / len(rev)
plt.figure(figsize=(6, 4)); plt.plot(share, cum.values)
plt.axvline(0.2, ls="--", c="grey"); plt.xlabel("share of customers (largest first)"); plt.ylabel("share of revenue")
plt.title("Cumulative revenue by customer"); plt.show()
print(f"Top 20% of customers: {cum.iloc[int(0.2 * len(rev)) - 1]:.0%} of revenue")

## 2. The churn rate moves with the season

Churn = no order in the next 90 days, among customers active in the last 180.

In [ ]:
rows = []
for c in config.BACKTEST_CUTOFFS:
    X, y = features.build_dataset(orders, c, 90, 180, "churn")
    rows.append({"cutoff": c, "active customers": len(y), "churn rate": y.mean()})
br = pd.DataFrame(rows)
plt.figure(figsize=(7, 3.5)); plt.bar(br["cutoff"], br["churn rate"]); plt.ylabel("churn rate"); plt.xticks(rotation=30)
plt.title("A model trained in one quarter sees a different base rate in the next"); plt.tight_layout(); plt.show()
br.round(3)

## 3. Health bands match what happened

Trained on one period, scored on the next, compared with the real outcome.

In [ ]:
scores, coef, n_lapsed = health.score(orders, "2011-03-01", "2011-06-01")
v = health.validate(orders, scores, "2011-06-01")
bands = v.index.astype(str)
plt.figure(figsize=(5, 3.5))
plt.bar(bands, v["observed_churn"], color=["tab:green", "tab:orange", "tab:red"], label="observed")
plt.plot(bands, v["mean_predicted"], "ko", label="predicted"); plt.ylabel("churn rate"); plt.legend(); plt.show()
v.round(3)

## 4. Evidence-aware ranking picks products that actually repeat

Out of sample: of buyers who had not yet repeated a product, the share who bought it again within 90 days.

In [ ]:
pc = pd.read_sql(f"select * from {config.PRODUCT_CUSTOMER_TABLE}", engine, parse_dates=["first_purchase_ts", "second_purchase_ts"])
pr = pd.read_sql(f"select stock_code, first_sale_ts from {config.PRODUCT_TABLE}", engine, parse_dates=["first_sale_ts"])
res = ranking_check.evaluate(pc, pr, pd.date_range("2010-07-01", "2011-09-01", freq="MS"), k=20)
summ = res.groupby("method")[["min_buyers_topk", "future_repeat"]].mean()
plt.figure(figsize=(5, 3.5)); plt.bar(summ.index, summ["future_repeat"]); plt.ylabel("repeat within 90 days"); plt.show()
summ.round(3)

## 5. Personas (behaviour only)

In [ ]:
f, prof, sil, k = personas.build_personas(orders, "2011-12-01", force_k=5)
plt.figure(figsize=(6, 4.5))
plt.scatter(np.log1p(f["monetary"]), f["recency_days"], c=f["persona"], s=6, cmap="tab10")
plt.xlabel("log(1 + total spend)"); plt.ylabel("days since last order"); plt.title(f"{k} behavioural personas"); plt.show()
prof.round(2)

Next: copy the figures that matter into the README, and rerun this notebook after any change to the cleaning rules.